In [ ]:
import os
import warnings
import pickle

from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, adjusted_mutual_info_score, balanced_accuracy_score
from sklearn.utils import shuffle

from catboost import CatBoostClassifier

from cstarters.ml_utils import avg, load_dataset, featurize_alignment, confusion_matrix_plot, plot_permutation_importances, plot_feature_importances, stdev_accuracy

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="sklearn"
)


random_seed = 2

In [ ]:
data_dir = "../../data"
out_dir = "../../out"

In [ ]:
df = load_dataset(path = os.path.join(data_dir, "cstarters_dataset.csv"))

aa_properties = {
    'WOLS870101': [924, 932, 944],
    'FAUJ880109': [991],
    'TSAJ990101': [814],
    'CHOP780203': [936, 818],
    'NEU1': [936],
    'NEU2': [98],
}

df = featurize_alignment(df, aa_properties)
df = df.dropna(subset=["starter_beta_hydroxylation"])
df = df[~df["starter_beta_hydroxylation"].isin(["unknown", "_not_elligible", "LCFA"])]
ml_df_full = df.drop(columns=["starter_category"])
ml_df_full = ml_df_full.drop(columns=['ncbi_organism_genus'])
ml_df_train, ml_df_test = train_test_split(ml_df_full, test_size = 0.2, stratify = df['starter_beta_hydroxylation'], random_state = random_seed)

y_train = ml_df_train['starter_beta_hydroxylation']
X_train = ml_df_train.drop(columns=['cs_accession', 'starter_beta_hydroxylation'])
X_train, y_train = shuffle(X_train, y_train, random_state=random_seed)

y_test = ml_df_test['starter_beta_hydroxylation']
X_test = ml_df_test.drop(columns=['cs_accession', 'starter_beta_hydroxylation'])
X_test, y_test = shuffle(X_test, y_test, random_state=random_seed)

y_full = ml_df_full['starter_beta_hydroxylation']
X_full = ml_df_full.drop(columns=['cs_accession', 'starter_beta_hydroxylation'])
X_full, y_full = shuffle(X_full, y_full, random_state=random_seed)


# Train

In [ ]:
# RF train
rf_params = dict(
    class_weight='balanced',
    ccp_alpha=0.02,
    max_depth=2,
    min_samples_leaf=2,
    max_leaf_nodes=3,
    min_samples_split=2,
    n_estimators=100,
    max_features="log2",
    random_state=random_seed
)
model = RandomForestClassifier(**rf_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • RF • Train Dataset")

In [ ]:
# SVM train
svm_params = dict(
    class_weight='balanced', 
    C=1, 
    degree=2, 
    gamma='auto', 
    kernel='rbf'
)
svm_params = dict(
    probability=True,
    C = 2.25, 
    class_weight='balanced',
    kernel='rbf',
    random_state=random_seed
)

model = svm.SVC(**svm_params)


cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • SVM • Train Dataset")

In [ ]:
# logistic regression train
lr_params = dict(
    solver='liblinear', 
    class_weight='balanced', 
    # penalty='l2', 
    C=1.04
)


model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • LR • Train Dataset")

In [ ]:
# catboost train
cat_params = dict(
    iterations=250,
    learning_rate=0.10,
    depth=3,
    l2_leaf_reg=8,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    random_seed=random_seed,
    verbose=0
)

model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_train, y_train, cv=5)
print(cv, "avg:", avg(cv))

cv_pred = cross_val_predict(model, X_train, y_train, cv=5)

confusion_matrix_plot(confusion_matrix(y_train, cv_pred),["FALSE", "TRUE"], title="β-Hydroxylation • CatBoost • Train Dataset")

# Test

In [ ]:
# RF test
model = RandomForestClassifier(**rf_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • RF • Test Dataset")

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# logistic regression test
model = LogisticRegression(**lr_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • LR • Test Dataset")


print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", adjusted_mutual_info_score(y_test, model_pred))

In [ ]:
# SVM test
model = svm.SVC(**svm_params)


model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • SVM • Test Dataset")
plot_permutation_importances(model, X_test, y_test)

print(classification_report(y_test, model_pred, digits=3))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

In [ ]:
# catboost test
model = CatBoostClassifier(**cat_params)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)

confusion_matrix_plot(confusion_matrix(y_test, model_pred), ["FALSE", "TRUE"], title="β-Hydroxylation • CatBoost • Test Dataset")
plot_feature_importances(model, X_test.columns, top_n=20)
print(classification_report(y_test, model_pred, digits=3,target_names=["FALSE", "TRUE"]))
print("adjusted mutual information\t", round(adjusted_mutual_info_score(y_test, model_pred), 3))
print("balanced accuracy\t", round(balanced_accuracy_score(y_test, model_pred), 3))

# Full

In [ ]:
# RF full
model = RandomForestClassifier(**rf_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="3-OH •  RF • Full Dataset")

In [ ]:
# logistic regression full
model = LogisticRegression(**lr_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="3-OH •  LogReg • Full Dataset")

stdev_accuracy(model, X_full, y_full, iterations=100)


In [ ]:
# SVM full
model = svm.SVC(**svm_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="3-OH •  SVM • Full Dataset")

stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
# catboost full
model = CatBoostClassifier(**cat_params)

cv = cross_val_score(model, X_full, y_full, cv=5)
print(cv, 'avg:',  avg(cv))
cv_pred = cross_val_predict(model, X_full, y_full, cv=5)

confusion_matrix_plot(confusion_matrix(y_full, cv_pred, labels=["FALSE", "TRUE"]), ["FALSE", "TRUE"], title="β-Hydroxylation • CatBoost • Full Dataset")
stdev_accuracy(model, X_full, y_full, iterations=100)

In [ ]:
# Previous model (retained for comparison):
# model = CatBoostClassifier(**cat_params)
# LR and SVM tie at 80% test accuracy; LR has higher balanced accuracy and macro-F1.
model = LogisticRegression(**lr_params)

model.fit(X_full, y_full)

pickle.dump(model, open(os.path.join(out_dir, "ml_hydroxylation.pkl"), "wb"))